# nanonis_reader Refactoring 테스트
---
June 3-4, 2026 작업 검증용 노트북

## 테스트 항목
1. Vendoring (nanonispy → _vendor)
2. Lazy properties (d.topo, d.spec, ...)
3. sweep_direction → method param
4. filter_sigma
5. image_processing 독립 함수
6. get_channel_name / has_averaged_data / find_sweep_channels
7. sweep_index 파라미터

In [2]:
import numpy as np
import nanonis_reader as nr
print(f'nanonis_reader loaded from: {nr.__file__}')

nanonis_reader loaded from: /Users/kim/Documents/git_clones/nanonis_reader/nanonis_reader/__init__.py


---
## 1. Vendoring 확인
`nanonispy`가 `_vendor`로 올바르게 vendoring 되었는지 확인.

In [3]:
from nanonis_reader import _vendor as nap
print(f'_vendor path: {nap.__file__}')
print(f'read module:  {nap.read}')
print('✅ Vendoring OK')

_vendor path: /Users/kim/Documents/git_clones/nanonis_reader/nanonis_reader/_vendor/__init__.py
read module:  <module 'nanonis_reader._vendor.read' from '/Users/kim/Documents/git_clones/nanonis_reader/nanonis_reader/_vendor/read.py'>
✅ Vendoring OK


---
## 2. Lazy Properties
### 2-1. .sxm 파일 테스트

In [33]:
# ⚠️ 경로를 실제 .sxm 파일로 수정하세요
sxm_path = '/Users/kim/Library/CloudStorage/OneDrive-Personal/Data/230922_TNS_STM6/TNS_STM6_#117_6492.sxm'

d = nr.load(sxm_path)
print(f'fname:   {d.fname}')
print(f'type:    {type(d.topo)}')
print(f'd.topo:  {d.topo}')
print(f'd.didv: {d.didv}')

# topo 처리 테스트
z_raw = d.topo.get_z(processing='raw')
z_lf  = d.topo.get_z(processing='subtract linear fit')
print(f'raw shape: {z_raw.shape}, linear fit shape: {z_lf.shape}')
print('✅ SXM lazy properties OK')

fname:   TNS_STM6_#117_6492.sxm
type:    <class 'nanonis_reader.sxm.topography'>
d.topo:  <nanonis_reader.sxm.topography object at 0x169c46f00>
d.didv: <nanonis_reader.sxm.didvmap object at 0x169b911d0>
raw shape: (128, 2048), linear fit shape: (128, 2048)
✅ SXM lazy properties OK


### 2-2. .dat 파일 테스트 (STS)

In [6]:
# ⚠️ 경로를 실제 STS .dat 파일로 수정하세요
dat_path = '/Users/kim/Library/CloudStorage/OneDrive-Personal/Data/230922_TNS_STM6/TNS_STM6_#117_0638.dat'

d = nr.load(dat_path)
print(f'd.spec: {d.spec}')

V, didv = d.spec.didv_scaled()
print(f'V shape: {V.shape}, dIdV shape: {didv.shape}')

# sweep_direction을 method에서 지정
V_bwd, didv_bwd = d.spec.didv_scaled(sweep_direction='bwd')
print(f'bwd dIdV shape: {didv_bwd.shape}')
print('✅ DAT lazy properties + sweep_direction OK')

d.spec: <nanonis_reader.dat.spectrum object at 0x10ff4dd30>
V shape: (201,), dIdV shape: (201,)
bwd dIdV shape: (201,)
✅ DAT lazy properties + sweep_direction OK


### 2-3. .3ds 파일 테스트 (Grid)

In [20]:
# ⚠️ 경로를 실제 .3ds 파일로 수정하세요
grid_path = '/Users/kim/Library/CloudStorage/OneDrive-Personal/Data/230922_TNS_STM6/TNS_STM6_#117_6355.3ds'

d = nr.load(grid_path)
print(f'd.topo:  {d.topo}')
print(f'd.point: {d.point}')
print(f'd.didv:  {d.didv}')

V, didv = d.point.get_didv_scaled(0, 0, channel='LI Demod 2 X')
print(f'Grid point dIdV shape: {didv.shape}')
print('✅ 3DS lazy properties OK')

d.topo:  <nanonis_reader.grid.topography object at 0x12f4d58c0>
d.point: <nanonis_reader.grid.point_didv object at 0x12f709220>
d.didv:  <nanonis_reader.grid.didvmap object at 0x12f3ab1d0>
Grid point dIdV shape: (141,)
✅ 3DS lazy properties OK


### 2-4. 잘못된 확장자에 대한 에러 확인

In [21]:
# .sxm 파일에서 .spec 접근 시 AttributeError 발생해야 함
d = nr.load(sxm_path)
try:
    _ = d.spec
    print('❌ Should have raised AttributeError')
except AttributeError as e:
    print(f'✅ 올바른 에러: {e}')

✅ 올바른 에러: .spec is not available for .sxm files (requires .dat)


---
## 3. filter_sigma (합성 데이터)

In [37]:
# 2D 스펙트럼 집합 (20개 스펙트럼, 100 bias points)
np.random.seed(42)
n_spectra, n_points = 20, 100
base_spectrum = np.sin(np.linspace(-2, 2, n_points))
data = np.tile(base_spectrum, (n_spectra, 1)) + np.random.normal(0, 0.05, (n_spectra, n_points))

# 2개의 outlier 추가
data[3] += 5.0   # 극단적 outlier
data[15] -= 5.0   # 극단적 outlier

filtered, mask = nr.filter_sigma(data, n_sigma=3)
print(f'원본: {data.shape[0]}개 → 필터 후: {filtered.shape[0]}개')
print(f'제거된 인덱스: {np.where(~mask)[0]}')
assert not mask[3] and not mask[15], 'Outliers should be removed'
print('✅ filter_sigma (unweighted) OK')

# weighted 모드
filtered_w, mask_w = nr.filter_sigma(data, n_sigma=3, weighted=True)
print(f'Weighted 필터 후: {filtered_w.shape[0]}개')
print(f'제거된 인덱스: {np.where(~mask_w)[0]}')
print('✅ filter_sigma (weighted) OK')

원본: 20개 → 필터 후: 18개
제거된 인덱스: [ 3 15]
✅ filter_sigma (unweighted) OK
Weighted 필터 후: 18개
제거된 인덱스: [ 3 15]
✅ filter_sigma (weighted) OK


---
## 4. image_processing 독립 함수 (합성 데이터)

In [14]:
from nanonis_reader import image_processing as ip

# 10x20 테스트 이미지: 선형 경사 + 노이즈
np.random.seed(0)
x = np.linspace(0, 5, 20)
z = np.tile(x, (10, 1)) + np.random.normal(0, 0.1, (10, 20))

# NaN 행 포함 (불완전 스캔 시뮬레이션)
z_nan = z.copy()
z_nan[8:] = np.nan

results = {}
for name, func in [
    ('subtract_average', ip.subtract_average),
    ('subtract_linear_fit', ip.subtract_linear_fit),
    ('subtract_linear_fit_xy', ip.subtract_linear_fit_xy),
    ('subtract_parabolic_fit', ip.subtract_parabolic_fit),
    ('subtract_plane_fit', ip.subtract_plane_fit),
]:
    result = func(z)
    result_nan = func(z_nan)
    results[name] = result
    
    assert result.shape == z.shape, f'{name}: shape mismatch'
    assert np.all(np.isnan(result_nan[8:])), f'{name}: NaN rows not preserved'
    print(f'{name}: shape={result.shape}, NaN preserved=True ✅')

# differentiate
deriv = ip.differentiate(z, dx=0.25)
assert deriv.shape == z.shape
print(f'differentiate: shape={deriv.shape} ✅')

print()
print('✅ All 6 image_processing functions OK')

subtract_average: shape=(10, 20), NaN preserved=True ✅
subtract_linear_fit: shape=(10, 20), NaN preserved=True ✅
subtract_linear_fit_xy: shape=(10, 20), NaN preserved=True ✅
subtract_parabolic_fit: shape=(10, 20), NaN preserved=True ✅
subtract_plane_fit: shape=(10, 20), NaN preserved=True ✅
differentiate: shape=(10, 20) ✅

✅ All 6 image_processing functions OK


### 4-1. sxm.topography가 image_processing을 호출하는지 확인

In [24]:
# 실제 .sxm 파일 필요
d = nr.load(sxm_path)

z_topo_lf = d.topo.get_z(processing='subtract linear fit')
z_direct_lf = ip.subtract_linear_fit(d.topo.raw('fwd'))

assert np.allclose(z_topo_lf, z_direct_lf, equal_nan=True)
print('✅ sxm.topography delegates to image_processing correctly')

✅ sxm.topography delegates to image_processing correctly


### 4-2. grid.topography가 image_processing을 호출하는지 확인

In [26]:
# 실제 .3ds 파일 필요
d = nr.load(grid_path)

z_topo_lf = d.topo.get_z(processing='subtract linear fit')
z_direct_lf = ip.subtract_linear_fit(d.topo.raw())

assert np.allclose(z_topo_lf, z_direct_lf, equal_nan=True)
print('✅ grid.topography delegates to image_processing correctly')

✅ grid.topography delegates to image_processing correctly


---
## 5. get_channel_name / has_averaged_data / find_sweep_channels

In [27]:
# 5-1. get_channel_name
assert nr.get_channel_name('LI Demod 1 X (A)') == 'LI Demod 1 X (A)'
assert nr.get_channel_name('LI Demod 1 X (A)', sweep_direction='bwd') == 'LI Demod 1 X [bwd] (A)'
assert nr.get_channel_name('LI Demod 1 X (A)', sweep_index=0) == 'LI Demod 1 X [00001] (A)'
assert nr.get_channel_name('LI Demod 1 X (A)', sweep_index=4) == 'LI Demod 1 X [00005] (A)'
assert nr.get_channel_name('LI Demod 1 X (A)', sweep_index=0, sweep_direction='bwd') == 'LI Demod 1 X [00001] [bwd] (A)'
assert nr.get_channel_name('Current (A)', sweep_direction='bwd') == 'Current [bwd] (A)'
print('✅ get_channel_name OK')

# 5-2. has_averaged_data
assert nr.has_averaged_data({'Current (A)': None, 'Current [AVG] (A)': None}) == True
assert nr.has_averaged_data({'Current (A)': None}) == False
print('✅ has_averaged_data OK')

# 5-3. find_sweep_channels
fake_signals = {
    'LI Demod 1 X (A)': None,
    'LI Demod 1 X [00001] (A)': None,
    'LI Demod 1 X [00002] (A)': None,
    'LI Demod 1 X [00003] (A)': None,
    'LI Demod 1 X [AVG] (A)': None,
    'LI Demod 1 X [00001] [bwd] (A)': None,
    'LI Demod 1 X [00002] [bwd] (A)': None,
    'LI Demod 1 X [AVG] [bwd] (A)': None,
    'Current (A)': None,
    'Current [00001] (A)': None,
    'Current [00002] (A)': None,
}

fwd_ch = nr.find_sweep_channels(fake_signals, 'LI Demod 1 X (A)', 'fwd')
bwd_ch = nr.find_sweep_channels(fake_signals, 'LI Demod 1 X (A)', 'bwd')
cur_ch = nr.find_sweep_channels(fake_signals, 'Current (A)', 'fwd')

print(f'fwd sweeps: {fwd_ch}')
print(f'bwd sweeps: {bwd_ch}')
print(f'Current fwd: {cur_ch}')

assert len(fwd_ch) == 3
assert len(bwd_ch) == 2
assert len(cur_ch) == 2
print('✅ find_sweep_channels OK')

✅ get_channel_name OK
✅ has_averaged_data OK
fwd sweeps: ['LI Demod 1 X [00001] (A)', 'LI Demod 1 X [00002] (A)', 'LI Demod 1 X [00003] (A)']
bwd sweeps: ['LI Demod 1 X [00001] [bwd] (A)', 'LI Demod 1 X [00002] [bwd] (A)']
Current fwd: ['Current [00001] (A)', 'Current [00002] (A)']
✅ find_sweep_channels OK


---
## 6. sweep_index 파라미터
### 6-1. dat.spectrum (실제 save-all 데이터 필요)

In [28]:
# ⚠️ save all이 적용된 .dat 파일 경로로 수정하세요
saveall_dat_path = '/Users/kim/Library/CloudStorage/OneDrive-Personal/Data/230922_TNS_STM6/TNS_STM6_#117_0638.dat'

d = nr.load(saveall_dat_path)
spec = d.spec

# 기본: AVG 데이터 반환 (1D)
V, didv_avg = spec.didv_scaled()
print(f'AVG: V={V.shape}, dIdV={didv_avg.shape}')

# 개별 sweep (0-indexed)
V, didv_0 = spec.didv_scaled(sweep_index=0)
print(f'sweep_index=0: V={V.shape}, dIdV={didv_0.shape}')

# 전체 sweep → stacked 2D
V, didv_all = spec.didv_scaled(sweep_index='all')
print(f'sweep_index=all: V={V.shape}, dIdV_all={didv_all.shape}')

print('✅ dat.spectrum sweep_index OK')

AVG: V=(201,), dIdV=(201,)
sweep_index=0: V=(201,), dIdV=(201,)
sweep_index=all: V=(201,), dIdV_all=(50, 201)
✅ dat.spectrum sweep_index OK


### 6-2. sweep_index='all' + filter_sigma 연계

In [29]:
# 위에서 얻은 didv_all 사용
V, didv_all = spec.didv_scaled(sweep_index='all')

filtered, mask = nr.filter_sigma(didv_all, n_sigma=3, weighted=True)
print(f'전체 {didv_all.shape[0]}개 → 필터 후 {filtered.shape[0]}개')
print(f'제거된 sweep 인덱스: {np.where(~mask)[0]}')

# 필터링 후 평균
didv_clean_mean = np.mean(filtered, axis=0)
print(f'Clean mean shape: {didv_clean_mean.shape}')
print('✅ sweep_index + filter_sigma pipeline OK')

전체 50개 → 필터 후 24개
제거된 sweep 인덱스: [ 0  4  6  7  9 10 11 12 15 18 19 20 21 24 26 27 28 29 30 31 33 36 38 40
 41 49]
Clean mean shape: (201,)
✅ sweep_index + filter_sigma pipeline OK


### 6-3. dat.z_spectrum

In [ ]:
# ⚠️ save all이 적용된 I(z) .dat 파일 경로로 수정하세요
saveall_iz_path = 'YOUR_SAVEALL_IZ_DAT_FILE.dat'

d = nr.load(saveall_iz_path)
iz = nr.dat.z_spectrum(d)

# 기본 (AVG)
z, I_avg = iz.get_iz()
print(f'AVG: z={z.shape}, I={I_avg.shape}')

# sweep_index='all'
z, I_all = iz.get_iz(sweep_index='all')
print(f'All: z={z.shape}, I_all={I_all.shape}')

# 하위 호환: sweep_dir='save all'
iz_compat = nr.dat.z_spectrum(d, sweep_direction='save all')
z, I_all_compat = iz_compat.get_iz()
print(f'Compat: I_all_compat={I_all_compat.shape}')

assert np.allclose(I_all, I_all_compat)
print('✅ z_spectrum sweep_index + backward compat OK')

### 6-4. grid.point_didv

In [31]:
# ⚠️ save all이 적용된 .3ds 파일 경로로 수정하세요
saveall_3ds_path = '/Users/kim/Library/CloudStorage/OneDrive-Personal/Data/230922_TNS_STM6/TNS_STM6_#117_0822.3ds'

d = nr.load(saveall_3ds_path)

# 기본 (AVG)
V, didv = d.point.get_didv_scaled(0, 0)
print(f'AVG: V={V.shape}, dIdV={didv.shape}')

# sweep_index='all'
V, didv_all = d.point.get_didv_scaled(0, 0, sweep_index='all')
print(f'All: V={V.shape}, dIdV_all={didv_all.shape}')

# 3-sigma
filtered, mask = nr.filter_sigma(didv_all, n_sigma=3, weighted=True)
print(f'Grid 3σ: {didv_all.shape[0]}개 → {filtered.shape[0]}개')
print('✅ grid.point_didv sweep_index OK')

AVG: V=(201,), dIdV=(201,)
All: V=(201,), dIdV_all=(3, 201)
Grid 3σ: 3개 → 3개
✅ grid.point_didv sweep_index OK


---
## 7. 중복 메서드 제거 확인

In [32]:
# 활성 클래스에서 get_channel_name / has_averaged_data 제거 확인
for cls_name, cls in [
    ('dat.spectrum', nr.dat.spectrum),
    ('grid.point_didv', nr.grid.point_didv),
    ('grid.point_iz', nr.grid.point_iz),
]:
    assert not hasattr(cls, 'get_channel_name'), f'{cls_name} still has get_channel_name'
    assert not hasattr(cls, 'has_averaged_data'), f'{cls_name} still has has_averaged_data'
    assert hasattr(cls, '_resolve_channel'), f'{cls_name} missing _resolve_channel'
    assert hasattr(cls, '_get_data'), f'{cls_name} missing _get_data'
    print(f'{cls_name}: old methods removed, new helpers present ✅')

print()
print('✅ 코드 중복 제거 완료 확인')

dat.spectrum: old methods removed, new helpers present ✅
grid.point_didv: old methods removed, new helpers present ✅
grid.point_iz: old methods removed, new helpers present ✅

✅ 코드 중복 제거 완료 확인


---
## Summary

| # | 항목 | 데이터 필요 |
|---|------|:-:|
| 1 | Vendoring | ❌ |
| 2 | Lazy properties | ✅ .sxm, .dat, .3ds |
| 3 | filter_sigma | ❌ (합성 데이터) |
| 4 | image_processing | ❌ (합성) + ✅ .sxm/.3ds |
| 5 | Channel utilities | ❌ (합성 데이터) |
| 6 | sweep_index | ✅ save-all 파일 |
| 7 | 중복 제거 확인 | ❌ |